In [21]:
import napari
from typing import List
from napari.utils import transforms as napari_transforms
import os
from pathlib import Path
import numpy as np
import glob
from dask_image import imread
import yaml
import json
import tqdm
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, Scale, Translation, Rotation, Affine, Sequence, Axis, Transform
from dataclasses import asdict
import zarr
import shutil
import transforms3d
from skimage import io
from dask.distributed import Client

from multiview_stitcher import msi_utils
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion

from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata

In [ ]:
client = Client()  # Dashboard at http://localhost:8787
print(f"Dashboard: {client.dashboard_link}")

Dashboard: http://127.0.0.1:8787/status


## Importing data

In [2]:
path_overview = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721\20250721_selectedPos_1_Focus_Plane_order2_1'

folders = [os.path.join(path_overview, folder) for folder in os.listdir(path_overview) if os.path.isdir(os.path.join(path_overview, folder))]
tif_files = glob.glob(os.path.join(path_overview, '**', '*.tif'))
metadata = yaml.safe_load(open(os.path.join(path_overview, folders[0], 'metadata.txt'), 'r'))['Summary']

positionlist = metadata['InitialPositionList']
positionlist = {position['Label']: position for position in positionlist}

## Stitching

In [3]:
tile_translations = []
tile_arrays = []
z_values = []
for label, position in tqdm.tqdm(positionlist.items()):
    z_scale = metadata['z-step_um'] if metadata['z-step_um'] != 0 else 1.0
    y_scale = metadata['PixelSize_um']
    x_scale = metadata['PixelSize_um']

    array = imread.imread(
        glob.glob(os.path.join(path_overview, label, '*.tif'))[0]
    )

    size_z = np.round(array.shape[0] * z_scale)
    size_y = np.round(array.shape[0] * y_scale)
    size_x = np.round(array.shape[1] * x_scale)

    tile_translations.append(
        {
            "y": -float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][1]) - size_y // 2,
            "x": float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][0]) - size_x // 2
            }
    )
    z_values.append(float(position["DeviceCoordinatesUm"]["ZStage:Z:32"][0]) - size_z // 2)
    tile_arrays.append(array)
z_offset = np.mean(z_values)

100%|██████████| 323/323 [00:03<00:00, 106.05it/s]


In [4]:
y_min = min([translation['y'] for translation in tile_translations])
x_min = min([translation['x'] for translation in tile_translations])
y_max = max([translation['y'] + size_y for translation in tile_translations])
x_max = max([translation['x'] + size_x for translation in tile_translations])
print(f"y_min: {y_min}, y_max: {y_max}, x_min: {x_min}, x_max: {x_max}, z_offset: {z_offset}")

y_min: 3198.7138671875, y_max: 10344.7138671875, x_min: 41801.08203125, x_max: 48699.08203125, z_offset: 1946.5478613885932


In [5]:
msims = []
for tile_array, tile_translation in zip(tile_arrays, tile_translations):
    sim = si_utils.get_sim_from_array(
        tile_array,
        dims=["c", "y", "x"],
        scale={"y": y_scale, "x": x_scale},
        translation=tile_translation,
        transform_key="stage_metadata",
    )
    msims.append(msi_utils.get_msim_from_sim(sim, scale_factors=[]))

fused_sim = fusion.fuse(
    [msi_utils.get_sim_from_msim(msim) for msim in msims],
    transform_key="stage_metadata"
).squeeze().pad(x=1, y=1, constant_values=0)

In [8]:
ngff_image_overview = NgffImage(
    fused_sim.data[None, :],
    dims=["z", "y", "x"],
    scale={"z": float(z_scale), "y": float(y_scale), "x": float(x_scale)},
    axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"},
    )

ngff_multiscales_overview = NgffMultiscales(
    ngff_image_overview,
    scale_factors=(2, 4, 8, 16, 32)
)

Dimension z is too small to downsample further.
Dimension z is too small to downsample further.
Dimension z is too small to downsample further.
Dimension z is too small to downsample further.
Dimension z is too small to downsample further.


## SCAPE storage

In [9]:
path_scape = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721\SCAPE\20250721_Pos1-2'

metadata_path = os.path.join(path_scape, '20250721_Pos1-2_MMStack_Pos0_metadata.txt')
with open(metadata_path, 'r', encoding='latin-1') as f:
    json_data = json.load(f)

x = float(json_data['Summary']['Position_X'].replace(',', '').replace(' µm', '').replace('µm', ''))
y = -float(json_data['Summary']['Position_Y'].replace(',', '').replace(' µm', '').replace('µm', ''))
z = float(json_data['Summary']['Position_SPIM_Head'].replace('µm', '').replace(',', ''))
angle = float(json_data['Summary']['StageScanAnglePathA'])
pixelsize_xy = float(json_data['Summary']['PixelSize_um'])
z_step = float(json_data['Summary']['z-step_um'])

slices_per_volume = int(json_data['Summary']['Slices'])

image_name = metadata_path.replace('_metadata.txt', '.ome.tif')
SCAPE_stack = io.imread(image_name)[0]

### Calculate affine matrix
image_name = metadata_path.replace('_metadata.txt', '.ome.tif')
SCAPE_stack = io.imread(image_name)[0]

size_y = SCAPE_stack.shape[1] * pixelsize_xy /np.cos(angle * np.pi / 180.0)
size_x = SCAPE_stack.shape[2] * pixelsize_xy
size_z = SCAPE_stack.shape[2] * z_step

shear_factor = np.cos(angle * np.pi / 180.0) * (z_step / pixelsize_xy)
new_dz = np.sin(angle * np.pi / 180.0) * z_step

shear_matrix = np.eye(3)
shear_matrix[1, 0] = shear_factor

affine_matrix = napari_transforms.Affine(
    shear=shear_matrix,
)
scale = np.array([z_step, pixelsize_xy, pixelsize_xy])
scale_correction = np.array([np.sin(angle * np.pi / 180.0), 1, 1])
translate_scape = [z, y + size_y / 2, x - size_x / 2]

# rotate to correct viewing angle
rotation_matrix = transforms3d.euler.euler2mat(angle, 0, 0)

In [26]:
ngff_image_scape = NgffImage(
    SCAPE_stack,
    dims=["z", "y", "x"],
    scale={"z": float(z_step), "y": float(pixelsize_xy), "x": float(pixelsize_xy)},
    axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"},
    )

ngff_multiscales_scape = NgffMultiscales(
    ngff_image_scape,
    scale_factors=(2, 4, 8)
)

# Create new coordinate systems tuple
new_cs_tuple = (
    ngff_multiscales_scape.metadata.coordinateSystems[0],
    CoordinateSystem(
        name='deskewed',
        axes=[
            Axis(name='z', type='space', unit='micrometer'),
            Axis(name='y', type='space', unit='micrometer'),
            Axis(name='x', type='space', unit='micrometer'),
        ]
    )
)
# Use model_copy() to update frozen metadata
ngff_multiscales_scape.metadata = ngff_multiscales_scape.metadata.model_copy(update={'coordinateSystems': new_cs_tuple})

intrinsic_cs = [cs for cs in ngff_multiscales_scape.metadata.coordinateSystems if cs.name == 'physical'][0]
deskewed_cs = [cs for cs in ngff_multiscales_scape.metadata.coordinateSystems if cs.name == 'deskewed'][0]

cs_input = CoordinateSystem(
    **intrinsic_cs.model_dump()
)
cs_output = CoordinateSystem(
    **deskewed_cs.model_dump()
)

rotation_ozm = Rotation(
    rotation=np.asarray(rotation_matrix).tolist(),
    name='rotation_transform',
)

center_rotation_translate = Translation(
            translation=(size_z / 2, size_y / 2, size_x / 2),
            name='center_translation',
        )
center_rotation_translate_inverse = Translation(
            translation=(- size_z / 2, - size_y / 2, - size_x / 2),
            name='center_translation_inverse',
        )

affine_ozm = Affine(
    affine=np.asarray(affine_matrix)[:-1, :].tolist(),
    name='deskewing_transform',
)

transformations: List[Transform] = [
    affine_ozm,
    center_rotation_translate,
    rotation_ozm,
    center_rotation_translate_inverse
    ]

deskewing_transform =Sequence(
    input=intrinsic_cs.name,
    output=deskewed_cs.name,
    transformations=transformations,
)

ngff_multiscales_scape.metadata = ngff_multiscales_scape.metadata.model_copy(
  update={'coordinateTransformations': [deskewing_transform.model_dump()]})

ngff_multiscales_scape.to_ome_zarr("test.ome.zarr")

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `tuple[TaggedUnion[definition-ref, definition-ref, definition-ref, definition-ref, definition-ref, definition-ref, definition-ref, definition-ref, definition-ref, definition-ref, definition-ref], ...]` - serialized value may not be as expected [field_name='coordinateTransformations', input_value=[{'type': 'sequence', 'in...00001), 'path': None})}], input_type=list])


## Scene creation

We will now create an instance of a scene collection.

In [11]:
cs_world = CoordinateSystem(
    name="world",
    axes=[
        Axis(name="z", type="space", unit="micrometer"),
        Axis(name="y", type="space", unit="micrometer"),
        Axis(name="x", type="space", unit="micrometer"),
    ],
)

overview_translation = Translation(
    translation=[float(z_offset), float(y_min), float(x_min)],
    output='world',
    input=InputOutput(
        coordinateSystem='intrinsic',
        path=Path(path_overview).stem,
    ),
    name='overview_translation',
)

scape_translation = Translation(
    translation=translate_scape,
    name='scape_translation',
    output='world',
    input=InputOutput(
        coordinateSystem='deskewed',
        path=Path(path_scape).stem,
    ),
)

In [ ]:
store_path = Path(path_overview).parent / 'scene_test.ome.zarr'

scene_metadata = SceneMetadata(
    coordinateSystems=[cs_world],
    coordinateTransformations=[
        overview_translation,
        scape_translation,
        ]
)

Scene = nz.Scene(
    store=store_path,
    scene_metadata=scene_metadata,
)

root = zarr.open_group(store_path, mode='a', zarr_format=3)
root.attrs['ome'] = asdict(scene_metadata)

# add the overview image
nz.to_ngff_zarr(store=str(store_path / Path(path_overview).stem), multiscales=ngff_multiscales_overview, version='0.5')
nz.to_ngff_zarr(store=str(store_path / Path('scape').stem), multiscales=ngff_multiscales_scape, version='0.5')

c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 36.57 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


is_vector: False

ITK Image:
Image(
    imageType=ImageType(dimension=3, componentType="uint16", pixelType="Scalar", components=1),
    name='Image',
    origin=[0.0, 0.0, 0.0],
    spacing=[1.0, 1.0, 1.0],
    direction=array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]),
    size=(0, 0, 0),
    metadata={},
    data=array([], shape=(0, 0, 0), dtype=uint16),
    bufferedRegion=ImageRegion(index=(0, 0, 0), size=(0, 0, 0))
)

c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 36.28 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [36]:
target = metadata_path.replace('_metadata.txt', '.ome.zarr')
nz.to_ngff_zarr(target, multiscales=ngff_multiscales, version="0.6")

c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 172.36 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


is_vector: False

ITK Image:
Image(
    imageType=ImageType(dimension=3, componentType="uint16", pixelType="Scalar", components=1),
    name='Image',
    origin=[0.0, 0.0, 0.0],
    spacing=[1.0, 1.0, 1.0],
    direction=array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]),
    size=(0, 0, 0),
    metadata={},
    data=array([], shape=(0, 0, 0), dtype=uint16),
    bufferedRegion=ImageRegion(index=(0, 0, 0), size=(0, 0, 0))
)

is_vector: False

ITK Image:
Image(
    imageType=ImageType(dimension=3, componentType="uint16", pixelType="Scalar", components=1),
    name='Image',
    origin=[0.0, 0.0, 0.0],
    spacing=[1.0, 1.0, 1.0],
    direction=array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]),
    size=(0, 0, 0),
    metadata={},
    data=array([], shape=(0, 0, 0), dtype=uint16),
    bufferedRegion=ImageRegion(index=(0, 0, 0), size=(0, 0, 0))
)

is_vector: False

ITK Image:
Image(
    imageType=ImageType(dimension=3, componentType="uint16", pixelType="Scalar", components=1),
    name='Image',
    origin=[0.0, 0.0, 0.0],
    spacing=[1.0, 1.0, 1.0],
    direction=array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]),
    size=(0, 0, 0),
    metadata={},
    data=array([], shape=(0, 0, 0), dtype=uint16),
    bufferedRegion=ImageRegion(index=(0, 0, 0), size=(0, 0, 0))
)

In [37]:
root = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721'

viewer.layers.clear()

layer_scape = viewer.open(
    os.path.join(root, r'SCAPE\20250721_Pos1-2\20250721_Pos1-2_MMStack_Pos0.ome.zarr'),
    plugin='napari-ome-zarr'
)[0]

# layer_overview = viewer.open(
#     os.path.join(root, '20250721_selectedPos_1_Focus_Plane_order2_1.ome.zarr'),
#     plugin='napari-ome-zarr'
# )[0]

layer_scape.bounding_box.visible = True

Root group {'ome': {'version': '0.6', 'multiscales': [{'coordinateSystems': [{'name': 'intrinsic', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer', 'orientation': None}, {'name': 'y', 'type': 'space', 'unit': 'micrometer', 'orientation': None}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}], 'coordinateTransformations': [{'type': 'sequence', 'input': 'intrinsic', 'output': 'deskewed', 'name': None, 'transformations': [{'type': 'affine', 'input': None, 'output': None, 'name': 'deskewing_transform', 'affine': [[1.0, 0.0, 0.0, 0.0], [0.8389501649494978, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0]], 'path': None}, {'type': 'translation', 'input': None, 'output': None, 'name': 'center_translation', 'translation': [150.0, 178.79488707059204, 117.30000000000001], 'path': None}, {'type': 'rotation', 'input': None, 'output': None, 'name': 'rotation_transform', 'rotation': [[1.0, -0.0, 0.0], [0.0, 0.3005925437436371, 0.9537526527594719], [-0.0, -0.9537526527594719, 0.3005925437436

## Display in neuroglancer

In [76]:
# full affine matrix
affine_matrix_ng = Affine(
    shear=shear_matrix,
)

In [77]:
viewer_n = neuroglancer.Viewer()
viewer_n

http://127.0.0.1:63891/v/bf8150ac00d8809d8c302b6ec9f6473ed297910b/

In [78]:
transform = neuroglancer.CoordinateSpaceTransform(
    output_dimensions=neuroglancer.CoordinateSpace(
        names=["z", "y", "x"],
        units="µm",
        scales=scale_correction * scale
    ),
    matrix=np.asarray(affine_matrix_ng)[:3]
)

source = neuroglancer.LayerDataSource(
    "zarr://http://192.168.178.64:3000/SCAPE/20250721_Pos1-2/20250721_Pos1-2_MMStack_Pos0.ome.zarr/",
    transform=transform
)

with viewer_n.txn() as s:
    s.layers['scape'] = neuroglancer.ImageLayer(
        source=source,
    )


In [43]:
with viewer_n.txn() as s:
    print(dir(s.layers['overview'].layer))

['VolumeRendering', '__annotations__', '__class__', '__deepcopy__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '_cached_wrappers', '_get_wrapped', '_json_data', '_lock', '_readonly', '_set_wrapped', 'annotationColor', 'annotation_color', 'blend', 'crossSectionRenderScale', 'cross_section_render_scale', 'interpolate', 'localDimensions', 'localPosition', 'localVelocity', 'local_dimensions', 'local_position', 'local_velocity', 'opacity', 'panels', 'pick', 'shader', 'shaderControls', 'shader_controls', 'source', 'supports_readonly', 'supports_validation', 'tab', 'to_json', 'tool', 'tool_bindings', 'type', 'volumeRenderingDepthSamples', 'volumeRenderingGain', 'volumeRenderingMode', 'volume_rendering', 'vol

## Store with ome-zarr-py

In [ ]:
root = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721'
path = os.path.join(root, 'scape_stack.ome.zarr')

store = parse_url(path, mode="w").store
root = zarr.group(store=store)

transformations = [
    {
        "type": "sequence",
        "input": "physical",
        "output": "world",
        "transformations": [
            {
                "type": "rotation",
                "rotation": np.asarray(rotation_matrix).tolist()
            },
            {
                "type": "affine",
                "affine": np.asarray(affine_matrix).tolist()
            },
            {
                "type": "scale",
                "scale": scale_correction.tolist()
            },
            {
                "type": "translate",
                "translation": translate
            }
        ]
    }
]

shutil.rmtree(path, ignore_errors=True)
write_image(
    SCAPE_stack,
    group=root,
    axes=['z', 'y', 'x'],
    scale=[z_step, pixelsize_xy, pixelsize_xy],
    )

print('Saved to ', path)

### Display in napari

In [ ]:
viewer.open(
    r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721\20250721_selectedPos_1_Focus_Plane_order2_1ome.zarr',
    plugin='napari-ome-zarr')

layer = viewer.add_image(
    SCAPE_stack,
    name='SCAPE Stack',
    contrast_limits=[0, 3000],
    colormap='bop blue',
    blending='additive',
    axis_labels=['z', 'y', 'x'],
    affine=affine_matrix,
)

layer.bounding_box.visible = True
viewer.camera.angles = (0, 0, 90)
viewer.camera.center = (0, y + size_y / 2, x - size_x / 2)

Root group {'ome': {'version': '0.5', 'multiscales': [{'axes': [{'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}], 'datasets': [{'path': 'scale0/image', 'coordinateTransformations': [{'scale': [0.3245, 0.3245], 'type': 'scale'}, {'translation': [3198.7138671875, 41801.08203125], 'type': 'translation'}]}, {'path': 'scale1/image', 'coordinateTransformations': [{'scale': [0.649, 0.649], 'type': 'scale'}, {'translation': [3198.8761171875, 41801.24428125], 'type': 'translation'}]}, {'path': 'scale2/image', 'coordinateTransformations': [{'scale': [1.298, 1.298], 'type': 'scale'}, {'translation': [3199.2006171875, 41801.568781250004], 'type': 'translation'}]}, {'path': 'scale3/image', 'coordinateTransformations': [{'scale': [2.596, 2.596], 'type': 'scale'}, {'translation': [3199.8496171875, 41802.21778125], 'type': 'translation'}]}, {'path': 'scale4/image', 'coordinateTransformations': [{'scale': [5.192, 5.192], 'type': 'scale'}, {'tra

## Object detection

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path='.runs/detect/train4/weights/yolo11n.pt',
    confidence_threshold=0.3,
    device="cuda:0",  # or 'cuda:0'
)

In [19]:
data = ngff_multiscales.images[0].data[1:-2, 1:-2]

# normalize to mean=126, std=64
data_normalized = (data - data.mean()) / data.std()  # standardize to mean=0, std=1
data = data_normalized * 64 + 126  # scale to mean=126, std=64
data = np.clip(data, 0, 255).astype(np.uint8)  # clip to valid range and convert to uint8
data = np.stack([data, data, data], axis=-1).compute().astype(np.uint8)

In [20]:
result = predict.get_prediction(data, detection_model=detection_model)

In [21]:
result.object_prediction_list

[]

In [ ]:
rectangles = []
features = pd.DataFrame()
for layer in viewer.layers:
    if not isinstance(layer, layers.Image):
        continue

    io.imsave('image.png', layer.data[0][0].compute())
    prediction = model('image.png')
    for obj_class, box, conf in zip(prediction[0].boxes.cls, prediction[0].boxes.xywhn, prediction[0].boxes.conf):
        cls = int(obj_class)
        box = box.detach().cpu().numpy()

        z0 = layer.translate[0]
        y0 = layer.translate[1]
        x0 = layer.translate[2]
        img_size = layer.data.shape * layer.scale
        y0_box = box[1] * img_size[1]
        x0_box = box[0] * img_size[2]
        w = box[2] * img_size[2]
        h = box[3] * img_size[1]

        rectangle_corners = np.array([
            [z0, y0 + y0_box - h/2, x0 + x0_box - w/2],
            [z0, y0 + y0_box - h/2, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box - w/2]
        ])
        rectangle_corners[:, 1:] += box[:2] * layer.scale[1:]

        _features = pd.DataFrame({
            'y0': [y0_box],
            'x0': [x0_box],
            'w': [w],
            'h': [h],
            'cls': [cls],
            'conf': [conf.detach().cpu().numpy()],
            'area': [w * h],
        })
        features = pd.concat([features, _features], ignore_index=True)

    rectangles.append(rectangle_corners)